# Export an existing nl2sh+ LoRA adapter (no retraining)

This notebook accepts an extracted-adapter ZIP, reads its declared base model, merges the adapter, exports Q4_K_M GGUF, and downloads the GGUF plus a SHA-256 manifest. It does not train or publish anything. Use a fresh Colab GPU runtime for the most reliable dependency setup. The training run did not record the base-model commit; the exporter records the resolved commit used now and states that limitation in the manifest.

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
else:
    raise RuntimeError('Select a Colab GPU runtime before exporting; no training will be run.')

## Upload the adapter ZIP

Upload the ZIP you downloaded after training (for example, `nl2sh-qlora-adapter.zip`). The notebook validates archive paths before extracting it.

In [ ]:
from google.colab import files
from pathlib import Path
from zipfile import ZipFile
import os, stat
uploaded = files.upload()
archives = [name for name in uploaded if name.lower().endswith('.zip')]
if len(archives) != 1:
    raise ValueError('Upload exactly one adapter ZIP file.')
adapter_root = Path('/content/nl2sh-adapter-upload').resolve()
adapter_root.mkdir(parents=True, exist_ok=True)
with ZipFile(archives[0]) as archive:
    for member in archive.infolist():
        target = (adapter_root / member.filename).resolve()
        if os.path.commonpath([str(adapter_root), str(target)]) != str(adapter_root):
            raise ValueError(f'Unsafe archive path: {member.filename}')
        if stat.S_ISLNK(member.external_attr >> 16):
            raise ValueError(f'Symlinks are not allowed in the adapter ZIP: {member.filename}')
    archive.extractall(adapter_root)
configs = list(adapter_root.rglob('adapter_config.json'))
if len(configs) != 1:
    raise ValueError(f'Expected one adapter_config.json; found {len(configs)}')
adapter = configs[0].parent
print('Adapter folder:', adapter)

In [ ]:
import subprocess, sys
repo = Path('/content/nl2sh-repo')
if not (repo / 'training' / 'export_adapter_gguf.py').is_file():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/HariHaran9597/nl2sh-Junior-Shell-Assistant.git', str(repo)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(repo / 'training' / 'export_requirements.txt')], check=True)
subprocess.run([sys.executable, '-c', "import torch, transformers, peft; from transformers.modeling_utils import PreTrainedModel; print('torch', torch.__version__, 'CUDA', torch.cuda.is_available()); print('transformers', transformers.__version__, 'peft', peft.__version__); print('PreTrainedModel import OK')"], check=True)
export_dir = Path('/content/nl2sh-export')
subprocess.run([sys.executable, '-u', str(repo / 'training' / 'export_adapter_gguf.py'), '--adapter', str(adapter), '--output-dir', str(export_dir)], check=True)
gguf = export_dir / 'nl2sh-q4_k_m.gguf'
manifest = export_dir / 'gguf_metadata.json'
assert gguf.is_file() and manifest.is_file(), 'GGUF export did not produce both expected files.'
print('GGUF:', gguf, f'({gguf.stat().st_size / 1024**2:.1f} MiB)')
print('Manifest:', manifest)

## Download the local-model files

Download both files. The GGUF is the model used by the CLI; the JSON manifest records its base revision and checksum. This export is not an accuracy evaluation and does not replace the current default model.

In [ ]:
files.download(str(gguf))
files.download(str(manifest))